# 03 — GFL/GFM Inverter Mode Classifier
## IEEE Submission Ready — 10-Point Reviewer Checklist

| # | Issue | Fix |
|---|---|---|
| 🔴 1 | GFL < 2% → degenerate | Multi-run generation: 15 runs × 3h, GFL ~8–10% |
| 🔴 2 | Data leakage | Scenario-based split: train B+C, test A+D |
| 🔴 3 | No ablation | Table III: base / temporal / full + causal delta |
| 🔴 4 | Unfair comparison | SVM-inst vs SVM+rolling (architecture-controlled) |
| 🟠 5 | AUC=1.000 undefended | Window sensitivity Table V + noise stress test |
| 🟠 6 | Cross-scenario not interpreted | Table IV with retention % and physical interpretation |
| 🟡 7 | Missing confusion matrix | Full confusion matrix with support column |
| 🟡 8 | SHAP not physically grounded | PLL/RoCoF interpretation added |
| 🟡 9 | Narrative = "applied RF" | Contribution = temporal feature representation |
| 🟢 10 | Reviewer objections | Noise stress, window sensitivity, limitation section |


In [ ]:
import sys, os, warnings, json, joblib
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0F172A'
matplotlib.rcParams['axes.facecolor']   = '#1E293B'
matplotlib.rcParams['text.color']       = '#F1F5F9'
matplotlib.rcParams['axes.labelcolor']  = '#F1F5F9'
matplotlib.rcParams['xtick.color']      = '#94A3B8'
matplotlib.rcParams['ytick.color']      = '#94A3B8'
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timezone, timedelta
from dataclasses import asdict
from scipy import stats

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, ConfusionMatrixDisplay)
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate)
try:
    from xgboost import XGBClassifier
    XGB_OK = True
except ImportError:
    XGB_OK = False
    print("WARNING: XGBoost not installed. Run: pip install xgboost")

from data_generator.ups_inverter_simulator import InverterSimulator

# ── Dataset A — Zone-A standard datacenter (48h, 6 inverters) ────────────────
# ── Dataset generation ────────────────────────────────────────────────────────
# islanding_probability=0.45 + 12 inverters + 14 days guarantees enough
# GFL and transitioning samples for stratified split (needs ≥2 per class).
# These parameters are documented in Section III of the paper.
sim_a = InverterSimulator(
    num_inverters=12,             # more inverters = more islanding events
    islanding_probability=0.45,   # raised to ensure rare classes have ≥10 samples
    datacenter_zone="ZONE-A",
    random_seed=42,
)
start = datetime(2024, 6, 1, 0, 0, tzinfo=timezone.utc)
records_a = []
for i in range(4032):         # 14 days × 24h × 12 steps/h
    ts = start + timedelta(minutes=5 * i)
    records_a.extend([asdict(r) for r in sim_a.generate_snapshot(ts)])

df_a = pd.DataFrame(records_a)
df_a['timestamp_utc'] = pd.to_datetime(df_a['timestamp_utc'], utc=True)
df_a['dataset'] = 'A'

print(f"Dataset A  shape : {df_a.shape}")
print("Class distribution:")
print(df_a['control_mode'].value_counts().to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET: Multi-run generation — guaranteed class balance
# ══════════════════════════════════════════════════════════════════════════════
# Physics: GFL state exists only at simulation START (fresh inverters begin GFL).
# Strategy: N short independent runs per scenario, each starting fresh.
# Each run captures the full GFL→transitioning→GFM trajectory.
#
# Scenario design (scenario-based split prevents leakage):
#   A (p=0.08, TEST ): GFL-dominant  — inverters rarely islanded
#   B (p=0.15, TRAIN): Mixed         — all 3 classes represented
#   C (p=0.35, TRAIN): Trans-heavy   — frequent mode switches
#   D (p=0.55, TEST ): GFM-dominant  — stressed grid, rapid transitions
#
# Split: Train B+C → Test A+D   (unseen extremes tested)
# ══════════════════════════════════════════════════════════════════════════════
from datetime import datetime, timezone, timedelta
from dataclasses import asdict
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from data_generator.ups_inverter_simulator import InverterSimulator
import warnings, logging
warnings.filterwarnings('ignore')

ALL_MODES = ['GFL', 'GFM', 'transitioning']

# Suppress verbose simulator logs
for _lgr in ['data_generator', 'data_generator.ups_inverter_simulator']:
    logging.getLogger(_lgr).setLevel(logging.CRITICAL)

N_RUNS  = 15   # independent simulations per scenario
N_STEPS = 36   # 3 hours × 12 steps/h per run

def generate_multi_run(p_island, zone, base_seed, n_runs=N_RUNS, n_steps=N_STEPS):
    # Function body:
    # N independent short simulations. Each starts fresh (GFL inverters).
    # Captures the full GFL → transitioning → GFM trajectory per run.
    records = []
    for run in range(n_runs):
        seed = base_seed + run * 7
        sim  = InverterSimulator(
            num_inverters=12,
            islanding_probability=p_island,
            datacenter_zone=zone,
            random_seed=seed,
        )
        start = datetime(2024, 6, 1 + (run % 28), 0, 0, tzinfo=timezone.utc)
        for i in range(n_steps):
            records.extend([asdict(r) for r in sim.generate_snapshot(
                start + timedelta(minutes=5 * i))])
    df = pd.DataFrame(records)
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
    return df

print(f"Generating 4 scenarios ({N_RUNS} runs × {N_STEPS} steps × 12 inverters)...")
print("  Scenario A (p=0.08, TEST)  ...", end=" ", flush=True)
df_scA = generate_multi_run(0.08, 'ZONE-A', base_seed=42);   print("done")
print("  Scenario B (p=0.15, TRAIN) ...", end=" ", flush=True)
df_scB = generate_multi_run(0.15, 'ZONE-B', base_seed=137);  print("done")
print("  Scenario C (p=0.35, TRAIN) ...", end=" ", flush=True)
df_scC = generate_multi_run(0.35, 'ZONE-C', base_seed=999);  print("done")
print("  Scenario D (p=0.55, TEST)  ...", end=" ", flush=True)
df_scD = generate_multi_run(0.55, 'ZONE-D', base_seed=314);  print("done")
df_scA['scenario']='A'; df_scB['scenario']='B'
df_scC['scenario']='C'; df_scD['scenario']='D'

print()
for name, df in [('A',df_scA),('B',df_scB),('C',df_scC),('D',df_scD)]:
    filt = df[df['control_mode'].isin(ALL_MODES)]
    total = len(filt)
    counts = filt['control_mode'].value_counts()
    parts = " | ".join(f"{k}={v}({v/total:.0%})" for k,v in counts.items())
    print(f"  Scenario {name} ({total:,} filtered): {parts}")


In [ ]:
# ── Feature Engineering + Scenario-Based Split ────────────────────────────────
BASE_FEATS = [
    'rocof_hz_per_s','thd_percent','freq_deviation_hz',
    'voltage_deviation_pu','output_active_power_kw','output_reactive_power_kvar',
    'output_voltage_v','output_frequency_hz','dc_link_voltage_v',
    'junction_temp_c','efficiency',
    # 'gfl_pll_locked' removed — label proxy / data leakage
]
ROLL_FEATS = [
    'rocof_roll_mean','rocof_roll_std','rocof_roll_max',
    'freq_roll_std','freq_roll_mean',
    'thd_roll_mean','power_roll_std','volt_roll_std',
]
ALL_FEATS  = BASE_FEATS + ROLL_FEATS
ROLL_WIN   = 4   # rolling window = 4 steps = 20 minutes

def engineer_features(df_raw, noise_seed=42, window=ROLL_WIN):
    df = df_raw[df_raw['control_mode'].isin(ALL_MODES)].copy()
    df = df.sort_values(['inverter_id','timestamp_utc'])
    grp = df.groupby('inverter_id')
    kw = dict(min_periods=1)
    df['rocof_roll_mean'] = grp['rocof_hz_per_s'].transform(lambda x: x.rolling(window,**kw).mean())
    df['rocof_roll_std']  = grp['rocof_hz_per_s'].transform(lambda x: x.rolling(window,**kw).std().fillna(0))
    df['rocof_roll_max']  = grp['rocof_hz_per_s'].transform(lambda x: x.rolling(window,**kw).max())
    df['freq_roll_std']   = grp['freq_deviation_hz'].transform(lambda x: x.rolling(window,**kw).std().fillna(0))
    df['freq_roll_mean']  = grp['freq_deviation_hz'].transform(lambda x: x.rolling(window,**kw).mean())
    df['thd_roll_mean']   = grp['thd_percent'].transform(lambda x: x.rolling(window,**kw).mean())
    df['power_roll_std']  = grp['output_active_power_kw'].transform(lambda x: x.rolling(window,**kw).std().fillna(0))
    df['volt_roll_std']   = grp['voltage_deviation_pu'].transform(lambda x: x.rolling(window,**kw).std().fillna(0))
    rng  = np.random.default_rng(noise_seed)
    Xraw = df[ALL_FEATS].fillna(0).astype(float).values
    df[ALL_FEATS] = Xraw + rng.normal(0, 0.05 * Xraw.std(axis=0), Xraw.shape)
    return df

df_A = engineer_features(df_scA, 42)
df_B = engineer_features(df_scB, 137)
df_C = engineer_features(df_scC, 999)
df_D = engineer_features(df_scD, 314)

le = LabelEncoder().fit(ALL_MODES)
def to_Xy(df_):
    return (df_[ALL_FEATS].fillna(0).astype(float).values,
            le.transform(df_['control_mode']))

X_A,y_A = to_Xy(df_A); X_B,y_B = to_Xy(df_B)
X_C,y_C = to_Xy(df_C); X_D,y_D = to_Xy(df_D)

# ── Scenario-based split (zero temporal leakage) ──────────────────────────────
# Train: B (mixed) + C (transitioning-heavy)   — seen operating conditions
# Test:  A (GFL-dominant) + D (stressed grid)  — unseen operating extremes
X_train = np.vstack([X_B, X_C]);  y_train = np.concatenate([y_B, y_C])
X_test  = np.vstack([X_A, X_D]);  y_test  = np.concatenate([y_A, y_D])

sc_main = StandardScaler()
X_train_s = sc_main.fit_transform(X_train)
X_test_s  = sc_main.transform(X_test)

SEEDS = [42, 7, 13, 99, 2024]

print(f"Features : {len(ALL_FEATS)} ({len(BASE_FEATS)} base + {len(ROLL_FEATS)} rolling, window={ROLL_WIN} steps = {ROLL_WIN*5} min)")
print(f"Train (B+C): {len(X_train):,}  |  Test (A+D): {len(X_test):,}")
print()
print("Class distribution — TEST SET (Scenarios A+D, unseen during training):")
for cls in le.classes_:
    enc = le.transform([cls])[0]
    cnt = (y_test == enc).sum()
    print(f"  {cls:15s}: {cnt:5,}  ({cnt/len(y_test):.1%})")
print()
print("Scenario-based split: no sample from test scenarios appears in training.")
print("Rolling features computed per-inverter within each scenario (no contamination).")


---
## Simulation Calibration — Formal Justification (Paper Section II-B)

This section documents the calibration of the simulation environment against
published IEEE standards and the NREL/UNIFI GFM specifications [R8].
It is the formal response to the reviewer critique:
*"Results are simulation-only and not validated against real hardware."*

**Calibration table:**

| Parameter | Simulated range | Standard | Reference |
|-----------|----------------|----------|-----------|
| ROCOF limit | ±2.0 Hz/s | Mandatory protection threshold | IEEE 1547-2018 §6.5.1 |
| THD (GFM mode) | ≤ 5% | Harmonic distortion at PCC | IEEE 519-2022 |
| Voltage deviation | ±0.10 p.u. | Ride-through voltage range | IEEE 1547-2018 Table 3 |
| VSM inertia constant H | 2–6 s | Physical SG range from literature | [R4] |
| VSM damping D | 10–30 | Physical SG range from literature | [R3] |
| SCR range | 1.2–3.0 | Stability boundary GFL at SCR≈1.5 | [R1] |
| Islanding detection limit | 2000 ms | Type I IBR requirement | IEEE 1547-2018 §6.5.1 |

**Paper paragraph (Section II-B — ready to paste into LaTeX):**

> The simulation environment is calibrated against IEEE 1547-2018 and the UNIFI
> Specifications for Grid-Forming Inverter-Based Resources [R8]. Specifically:
> (i) ROCOF is bounded to ±2 Hz/s, consistent with the mandatory protection
> threshold of IEEE 1547-2018 §6.5.1 for Type I IBRs; (ii) THD is constrained
> to ≤5% in GFM mode, per IEEE 519-2022 at the point of common coupling;
> (iii) voltage deviation is within ±0.10 p.u., matching IEEE 1547-2018 Table 3;
> (iv) VSM swing equation parameters — H ∈ [2,6] s and D ∈ [10,30] — are within
> the range reported for physical synchronous generators [R3],[R4]; (v) SCR spans
> 1.2–3.0, covering the GFL stability boundary at SCR≈1.5 identified in [R1].
> While simulation-based evaluation cannot substitute for field validation,
> the cross-scenario protocol (Section IV-C) provides a lower bound on
> generalization under real-world distribution shift.


In [ ]:
# ── Method 0: ROCOF-Threshold (IEEE 1547) ────────────────────────────────────
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# Test set ROCOF values from combined A+D dataframes
_df_test = pd.concat(
    [df_A[df_A['control_mode'].isin(ALL_MODES)],
     df_D[df_D['control_mode'].isin(ALL_MODES)]], ignore_index=True
).reset_index(drop=True)
rocof_te = np.abs(_df_test['rocof_hz_per_s'].values[:len(y_test)])
y_te_str = le.inverse_transform(y_test)

def rocof_predict(rocof_abs, thr_low, thr_high):
    return np.where(rocof_abs <= thr_low, 'GFM',
                    np.where(rocof_abs >= thr_high, 'GFL', 'transitioning'))

# Grid-search optimal thresholds (best-case ROCOF — fair upper bound)
best_f1_thr, best_cfg = 0.0, (0.0, 0.0)
for tl in np.arange(0.03, 0.40, 0.02):
    for th in np.arange(0.40, 2.0, 0.05):
        if th <= tl:
            continue
        f1 = f1_score(y_te_str, rocof_predict(rocof_te, tl, th),
                      average='macro', zero_division=0)
        if f1 > best_f1_thr:
            best_f1_thr = f1
            best_cfg = (round(tl, 2), round(th, 2))

thr_low_best, thr_high_best = best_cfg
preds_rocof = rocof_predict(rocof_te, thr_low_best, thr_high_best)

rep_rocof = classification_report(y_te_str, preds_rocof,
    labels=le.classes_, target_names=le.classes_,
    output_dict=True, zero_division=0)
rocof_f1_macro    = rep_rocof['macro avg']['f1-score']
rocof_f1_by_class = {c: rep_rocof.get(c, {}).get('f1-score', 0.0) for c in le.classes_}
rocof_support     = {c: int(rep_rocof.get(c, {}).get('support', 0))  for c in le.classes_}

print(f"ROCOF thresholds: low={thr_low_best} Hz/s → GFM | high={thr_high_best} Hz/s → GFL")
print()
print(classification_report(y_te_str, preds_rocof,
    labels=le.classes_, target_names=le.classes_, zero_division=0))
print(f"Support: GFL={rocof_support['GFL']}  GFM={rocof_support['GFM']}  "
      f"transitioning={rocof_support['transitioning']}")


In [ ]:
# ── Methods 1a & 1b: SVM-inst vs SVM+rolling (architecture-controlled) ────────
# FIX 4: If SVM+rolling ≈ RF, temporal features drive improvement — not RF arch.
# This is the key causal experiment: same model, different feature sets.
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, classification_report

def add_derived(df_):
    df_out = df_.copy()
    df_out['rocof_norm']  = (df_out['rocof_hz_per_s'].abs() /
                              df_out['freq_deviation_hz'].abs().clip(lower=0.001)).clip(0, 100)
    df_out['dv_df_ratio'] = (df_out['voltage_deviation_pu'].abs() /
                              df_out['freq_deviation_hz'].abs().clip(lower=0.001)).clip(0, 200)
    return df_out

df_B2 = add_derived(df_B); df_C2 = add_derived(df_C)
df_A2 = add_derived(df_A); df_D2 = add_derived(df_D)

SVM_INST = [
    'rocof_hz_per_s','freq_deviation_hz','output_frequency_hz',
    'voltage_deviation_pu','output_voltage_v','thd_percent',
    'output_active_power_kw','output_reactive_power_kvar',
    'rocof_norm','dv_df_ratio'
]
SVM_ROLL = ALL_FEATS + ['rocof_norm', 'dv_df_ratio']

svm_results = {
    'inst': {k: [] for k in ['auc','f1_macro','f1_GFL','f1_GFM','f1_trans']},
    'roll': {k: [] for k in ['auc','f1_macro','f1_GFL','f1_GFM','f1_trans']},
}

for variant, feats in [('inst', SVM_INST), ('roll', SVM_ROLL)]:
    # Train B+C, Test A+D — consistent with main split
    Xtr_svm = np.vstack([df_B2[feats].fillna(0).astype(float).values,
                          df_C2[feats].fillna(0).astype(float).values])
    Xte_svm = np.vstack([df_A2[feats].fillna(0).astype(float).values,
                          df_D2[feats].fillna(0).astype(float).values])
    y_tr_svm = y_train
    y_te_svm = y_test
    print(f"SVM-{variant} ({len(feats)} feats) — 5 seeds...")
    for seed in SEEDS:
        # Stratified subsample for SVM speed
        n_sub = min(5000, len(Xtr_svm))
        _, idx_sub = train_test_split(
            np.arange(len(Xtr_svm)),
            test_size=n_sub / len(Xtr_svm),
            stratify=y_tr_svm, random_state=seed)
        sc_sv = StandardScaler()
        Xtr_s = sc_sv.fit_transform(Xtr_svm[idx_sub])
        Xte_s = sc_sv.transform(Xte_svm)
        svm = SVC(kernel='rbf', C=10.0, gamma='scale',
                  class_weight='balanced', probability=True, random_state=seed)
        svm.fit(Xtr_s, y_tr_svm[idx_sub])
        ypred = svm.predict(Xte_s)
        yprob = svm.predict_proba(Xte_s)
        try:
            auc = roc_auc_score(y_te_svm, yprob, multi_class='ovr', average='macro',
                                labels=list(range(len(le.classes_))))
        except ValueError:
            auc = float('nan')
        rep = classification_report(y_te_svm, ypred,
            labels=list(range(len(le.classes_))), target_names=le.classes_,
            output_dict=True, zero_division=0)
        svm_results[variant]['auc'].append(auc)
        svm_results[variant]['f1_macro'].append(rep['macro avg']['f1-score'])
        svm_results[variant]['f1_GFL'].append(rep['GFL']['f1-score'])
        svm_results[variant]['f1_GFM'].append(rep['GFM']['f1-score'])
        svm_results[variant]['f1_trans'].append(rep['transitioning']['f1-score'])

svm_mean = {v: {k: (np.mean(vals), np.std(vals)) for k, vals in res.items()}
            for v, res in svm_results.items()}

for v in ['inst', 'roll']:
    m = svm_mean[v]
    print(f"  SVM-{v}: AUC={m['auc'][0]:.4f}  F1={m['f1_macro'][0]:.4f}  "
          f"F1-trans={m['f1_trans'][0]:.4f}")

delta_svm = svm_mean['roll']['f1_macro'][0] - svm_mean['inst']['f1_macro'][0]
delta_trans = svm_mean['roll']['f1_trans'][0] - svm_mean['inst']['f1_trans'][0]
print()
print(f"Architecture-controlled delta (SVM-roll vs SVM-inst):")
print(f"  ΔF1-macro:        {delta_svm:+.4f}")
print(f"  ΔF1-transitioning: {delta_trans:+.4f}")
print()
if delta_svm >= 0.10:
    print("→ STRONG: temporal features drive improvement (not RF architecture)")
elif delta_svm >= 0.05:
    print("→ MODERATE: temporal features contribute; architecture also matters")
else:
    print("→ WEAK: reframe contribution around RF ensemble properties")


In [ ]:
# ── Method 2: XGBoost ────────────────────────────────────────────────────────
from sklearn.metrics import roc_auc_score, classification_report
try:
    from xgboost import XGBClassifier
    XGB_OK = True
except ImportError:
    XGB_OK = False
    print("pip install xgboost")

if not XGB_OK:
    xgb_mean = {k: (0, 0) for k in ['auc','f1_macro','f1_GFL','f1_GFM','f1_trans']}
else:
    xgb_res = {k: [] for k in ['auc','f1_macro','f1_GFL','f1_GFM','f1_trans']}
    print("XGBoost — 5 seeds (B+C train → A+D test)...")
    for seed in SEEDS:
        sc_x = StandardScaler()
        Xtr_s = sc_x.fit_transform(X_train)
        Xte_s = sc_x.transform(X_test)
        xgb = XGBClassifier(
            n_estimators=400, max_depth=7, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
            use_label_encoder=False, eval_metric='mlogloss',
            random_state=seed, n_jobs=-1, verbosity=0)
        xgb.fit(Xtr_s, y_train)
        ypred = xgb.predict(Xte_s)
        yprob = xgb.predict_proba(Xte_s)
        try:
            auc = roc_auc_score(y_test, yprob, multi_class='ovr', average='macro',
                                labels=list(range(len(le.classes_))))
        except ValueError:
            auc = float('nan')
        rep = classification_report(y_test, ypred,
            labels=list(range(len(le.classes_))), target_names=le.classes_,
            output_dict=True, zero_division=0)
        xgb_res['auc'].append(auc)
        xgb_res['f1_macro'].append(rep['macro avg']['f1-score'])
        xgb_res['f1_GFL'].append(rep['GFL']['f1-score'])
        xgb_res['f1_GFM'].append(rep['GFM']['f1-score'])
        xgb_res['f1_trans'].append(rep['transitioning']['f1-score'])
        print(f"  Seed {seed:>5}: AUC={auc:.4f}  F1={rep['macro avg']['f1-score']:.4f}  "
              f"F1-trans={rep['transitioning']['f1-score']:.4f}")
    xgb_mean = {k: (np.mean(v), np.std(v)) for k, v in xgb_res.items()}
    print(f"  XGB: AUC={xgb_mean['auc'][0]:.4f}  F1={xgb_mean['f1_macro'][0]:.4f}")


In [ ]:
# ── Method 3: Random Forest + Temporal Rolling Features (Proposed) ───────────
from sklearn.ensemble import RandomForestClassifier

rf_res = {k: [] for k in ['auc','f1_macro','f1_GFL','f1_GFM','f1_trans','preds']}
print("RF (proposed, B+C train → A+D test) — 5 seeds...")
for seed in SEEDS:
    sc_r = StandardScaler()
    Xtr_s = sc_r.fit_transform(X_train)
    Xte_s = sc_r.transform(X_test)
    rf = RandomForestClassifier(
        n_estimators=400, class_weight='balanced',
        max_depth=18, min_samples_leaf=3,
        max_features='sqrt', random_state=seed, n_jobs=-1)
    rf.fit(Xtr_s, y_train)
    ypred = rf.predict(Xte_s)
    yprob = rf.predict_proba(Xte_s)
    try:
        auc = roc_auc_score(y_test, yprob, multi_class='ovr', average='macro',
                            labels=list(range(len(le.classes_))))
    except ValueError:
        auc = float('nan')
    rep = classification_report(y_test, ypred,
        labels=list(range(len(le.classes_))), target_names=le.classes_,
        output_dict=True, zero_division=0)
    rf_res['auc'].append(auc)
    rf_res['f1_macro'].append(rep['macro avg']['f1-score'])
    rf_res['f1_GFL'].append(rep['GFL']['f1-score'])
    rf_res['f1_GFM'].append(rep['GFM']['f1-score'])
    rf_res['f1_trans'].append(rep['transitioning']['f1-score'])
    rf_res['preds'].append(ypred)
    print(f"  Seed {seed:>5}: AUC={auc:.4f}  F1={rep['macro avg']['f1-score']:.4f}  "
          f"F1-trans={rep['transitioning']['f1-score']:.4f}")

rf_mean = {k: (np.mean(v), np.std(v)) for k, v in rf_res.items() if k != 'preds'}
print(f"\n  RF: AUC={rf_mean['auc'][0]:.4f}±{rf_mean['auc'][1]:.4f}  "
      f"F1={rf_mean['f1_macro'][0]:.4f}±{rf_mean['f1_macro'][1]:.4f}  "
      f"F1-trans={rf_mean['f1_trans'][0]:.4f}±{rf_mean['f1_trans'][1]:.4f}")


In [ ]:
# ── Table II — Full Comparison + Confusion Matrix (FIX 7) ───────────────────
from scipy import stats
from sklearn.metrics import confusion_matrix

print("=" * 92)
print("TABLE II — CLASSIFIER COMPARISON  (train B+C → test A+D, 5 seeds)")
print("=" * 92)
print(f"{'Method':<24}  {'Feats':>5}  {'AUC':>14}  {'F1-macro':>14}  "
      f"{'F1-GFL':>8}  {'F1-GFM':>8}  {'F1-trans':>10}")
print("-" * 92)

supp = f"n={len(y_test):,} | GFL={rocof_support['GFL']} GFM={rocof_support['GFM']} trans={rocof_support['transitioning']}"

# ROCOF row
print(f"{'ROCOF-thr (IEEE 1547)':<24}  {'1':>5}  {'—':>14}  "
      f"{rocof_f1_macro:14.4f}  "
      f"{rocof_f1_by_class['GFL']:8.4f}  "
      f"{rocof_f1_by_class['GFM']:8.4f}  "
      f"{rocof_f1_by_class['transitioning']:10.4f}")

for vname, vkey, nf in [('SVM-inst (lit.)', 'inst', len(SVM_INST)),
                         ('SVM+rolling (fair)', 'roll', len(SVM_ROLL))]:
    m = svm_mean[vkey]
    print(f"{vname:<24}  {nf:>5}  "
          f"{m['auc'][0]:.4f}±{m['auc'][1]:.4f}  "
          f"{m['f1_macro'][0]:.4f}±{m['f1_macro'][1]:.4f}  "
          f"{m['f1_GFL'][0]:.4f}±{m['f1_GFL'][1]:.4f}  "
          f"{m['f1_GFM'][0]:.4f}±{m['f1_GFM'][1]:.4f}  "
          f"{m['f1_trans'][0]:.4f}±{m['f1_trans'][1]:.4f}")

if XGB_OK:
    m = xgb_mean
    print(f"{'XGBoost':<24}  {len(ALL_FEATS):>5}  "
          f"{m['auc'][0]:.4f}±{m['auc'][1]:.4f}  "
          f"{m['f1_macro'][0]:.4f}±{m['f1_macro'][1]:.4f}  "
          f"{m['f1_GFL'][0]:.4f}±{m['f1_GFL'][1]:.4f}  "
          f"{m['f1_GFM'][0]:.4f}±{m['f1_GFM'][1]:.4f}  "
          f"{m['f1_trans'][0]:.4f}±{m['f1_trans'][1]:.4f}")

m = rf_mean
print(f"{'RF+rolling (proposed)':<24}  {len(ALL_FEATS):>5}  "
      f"{m['auc'][0]:.4f}±{m['auc'][1]:.4f}  "
      f"{m['f1_macro'][0]:.4f}±{m['f1_macro'][1]:.4f}  "
      f"{m['f1_GFL'][0]:.4f}±{m['f1_GFL'][1]:.4f}  "
      f"{m['f1_GFM'][0]:.4f}±{m['f1_GFM'][1]:.4f}  "
      f"{m['f1_trans'][0]:.4f}±{m['f1_trans'][1]:.4f}")

print("=" * 92)
print(f"Support: {supp}  |  mean±std over seeds {SEEDS}")
print()

# Key narrative metrics
rf_f1 = rf_mean['f1_macro'][0]
print(f"Key paper metrics:")
print(f"  RF vs ROCOF-thr:    ΔF1-macro={rf_f1-rocof_f1_macro:+.4f}  "
      f"ΔF1-trans={rf_mean['f1_trans'][0]-rocof_f1_by_class['transitioning']:+.4f}")
print(f"  SVM-roll vs SVM-inst: ΔF1-macro={svm_mean['roll']['f1_macro'][0]-svm_mean['inst']['f1_macro'][0]:+.4f}  "
      f"ΔF1-trans={svm_mean['roll']['f1_trans'][0]-svm_mean['inst']['f1_trans'][0]:+.4f}")
print(f"  RF vs SVM+rolling:  ΔF1-macro={rf_f1-svm_mean['roll']['f1_macro'][0]:+.4f}")

# ── Confusion matrix (seed=42) with support ───────────────────────────────────
print()
print("Confusion Matrix — RF seed=42 (rows=true, cols=predicted):")
cm = confusion_matrix(y_test, rf_res['preds'][0], labels=list(range(len(le.classes_))))
print(f"{'':20s}  " + "  ".join(f"{c:>14s}" for c in le.classes_))
for i, cls in enumerate(le.classes_):
    row = "  ".join(f"{cm[i,j]:>14,}" for j in range(len(le.classes_)))
    print(f"  True {cls:13s}  {row}   support={cm[i].sum()}")

# McNemar RF vs XGB
if XGB_OK:
    sc_mc = StandardScaler()
    Xtr_mc = sc_mc.fit_transform(X_train); Xte_mc = sc_mc.transform(X_test)
    rf_mc = RandomForestClassifier(n_estimators=400, class_weight='balanced',
        max_depth=18, min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)
    rf_mc.fit(Xtr_mc, y_train)
    xgb_mc = XGBClassifier(n_estimators=400, max_depth=7, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, use_label_encoder=False,
        eval_metric='mlogloss', random_state=42, n_jobs=-1, verbosity=0)
    xgb_mc.fit(Xtr_mc, y_train)
    yrf = rf_mc.predict(Xte_mc); yxgb = xgb_mc.predict(Xte_mc)
    b = int(np.sum((yrf==y_test) & (yxgb!=y_test)))
    c_ = int(np.sum((yrf!=y_test) & (yxgb==y_test)))
    chi2 = (abs(b-c_)-1)**2 / max(b+c_, 1)
    pval = 1 - stats.chi2.cdf(chi2, df=1)
    print(f"\nMcNemar RF vs XGBoost: b={b}, c={c_}, χ²={chi2:.3f}, p={pval:.4f}")
    if pval < 0.05:
        print("  → Significant: RF significantly outperforms XGBoost (α=0.05)")
    else:
        print("  → No significant difference. Paper framing: RF preferred for")
        print("    interpretability (SHAP TreeExplainer), not accuracy superiority.")


In [ ]:
# ── Wilcoxon Signed-Rank Test + Bootstrap CI (Statistical Credibility) ────────
# Required by IEEE Transactions: statistical tests between methods
from scipy import stats
import numpy as np

# Five-seed F1-macro scores for each method
# Fill from NB03 actual outputs (these are placeholders — update after run)
f1_rf_seeds    = rf_res['f1_macro']           # list of 5 values
f1_svm_roll    = svm_results['roll']['f1_macro']
f1_svm_inst    = svm_results['inst']['f1_macro']
f1_xgb_seeds   = xgb_res['f1_macro'] if XGB_OK else [0]*5

# Wilcoxon signed-rank test (one-sided: alternative='greater')
def wilcoxon_test(a, b, label):
    try:
        stat, p = stats.wilcoxon(a, b, alternative='greater')
        sig = "SIGNIFICANT" if p < 0.05 else "not significant"
        print(f"  {label}: W={stat:.0f}, p={p:.3f} -> {sig} (alpha=0.05)")
        return stat, p
    except Exception as e:
        print(f"  {label}: cannot compute ({e})")
        return None, None

print("Wilcoxon Signed-Rank Tests (H1: row > column):")
w1, p1 = wilcoxon_test(f1_svm_roll, f1_svm_inst,
                        "SVM+rolling > SVM-inst  (architecture-controlled)")
w2, p2 = wilcoxon_test(f1_rf_seeds, f1_svm_roll,
                        "RF > SVM+rolling        (model architecture benefit)")
if XGB_OK:
    w3, p3 = wilcoxon_test(f1_rf_seeds, f1_xgb_seeds,
                            "RF > XGBoost            (ensemble comparison)")

print()
print("Paper narrative:")
print(f"  SVM+rolling vs SVM-inst: p={p1:.3f} -> rolling features {'significantly' if p1 and p1<0.05 else 'not significantly'} better")
print(f"  RF vs SVM+rolling:       p={p2:.3f} -> model architecture {'significantly' if p2 and p2<0.05 else 'not significantly'} better")

# Bootstrap 95% CI on RF seed=42 F1-macro
print()
print("Bootstrap 95% CI (n=1000 resamples, RF seed=42):")
rng_bs = np.random.default_rng(42)
sc_bs  = __import__('sklearn.preprocessing', fromlist=['StandardScaler']).StandardScaler()
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

sc_b = __import__('sklearn.preprocessing', fromlist=['StandardScaler']).StandardScaler()
X_tr_b = sc_b.fit_transform(X_train)
X_te_b  = sc_b.transform(X_test)
rf_b = RandomForestClassifier(n_estimators=400, class_weight='balanced',
    max_depth=18, min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)
rf_b.fit(X_tr_b, y_train)
y_pred_b = rf_b.predict(X_te_b)

boot_f1 = []
n_test   = len(y_test)
for _ in range(1000):
    idx_b = rng_bs.integers(0, n_test, n_test)
    boot_f1.append(f1_score(y_test[idx_b], y_pred_b[idx_b],
        average='macro', zero_division=0,
        labels=list(range(len(le.classes_)))))

ci_lo, ci_hi = np.percentile(boot_f1, [2.5, 97.5])
print(f"  RF seed=42: F1-macro = {np.mean(boot_f1):.4f}  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
print()
print("Add to paper Table II footnote:")
print(f"  95% CI (bootstrap, n=1000): [{ci_lo:.4f}, {ci_hi:.4f}]")


In [ ]:
# ── Table III — Ablation Study ────────────────────────────────────────────────
# CRITICAL: proves temporal features are the causal contributor, not RF arch.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

ablation_configs = {
    'RF-base (11 feats)':     BASE_FEATS,
    'RF-temporal (8 feats)':  ROLL_FEATS,
    'RF-full (19 feats)':     ALL_FEATS,
}
ablation_results = {}
print("Ablation study — causal contribution of temporal features (5 seeds each)...")
for name, feats in ablation_configs.items():
    # B+C train, A+D test — same split as main experiment
    Xtr_ab = np.vstack([df_B[feats].fillna(0).astype(float).values,
                         df_C[feats].fillna(0).astype(float).values])
    Xte_ab = np.vstack([df_A[feats].fillna(0).astype(float).values,
                         df_D[feats].fillna(0).astype(float).values])
    results = {k: [] for k in ['auc','f1_macro','f1_trans']}
    for seed in SEEDS:
        sc_ab = StandardScaler()
        Xtr_s = sc_ab.fit_transform(Xtr_ab)
        Xte_s = sc_ab.transform(Xte_ab)
        rf_ab = RandomForestClassifier(n_estimators=200, class_weight='balanced',
            max_depth=18, min_samples_leaf=3, max_features='sqrt',
            random_state=seed, n_jobs=-1)
        rf_ab.fit(Xtr_s, y_train)
        ypred = rf_ab.predict(Xte_s)
        yprob = rf_ab.predict_proba(Xte_s)
        try:
            auc = roc_auc_score(y_test, yprob, multi_class='ovr', average='macro',
                                labels=list(range(len(le.classes_))))
        except ValueError:
            auc = float('nan')
        rep = classification_report(y_test, ypred,
            labels=list(range(len(le.classes_))), target_names=le.classes_,
            output_dict=True, zero_division=0)
        results['auc'].append(auc)
        results['f1_macro'].append(rep['macro avg']['f1-score'])
        results['f1_trans'].append(rep['transitioning']['f1-score'])
    ablation_results[name] = {k: (np.mean(v), np.std(v)) for k, v in results.items()}
    print(f"  {name:<26}: F1={ablation_results[name]['f1_macro'][0]:.4f}  "
          f"F1-trans={ablation_results[name]['f1_trans'][0]:.4f}")

print()
print("=" * 70)
print("TABLE III — ABLATION (RF variants, 5 seeds, train B+C → test A+D)")
print("=" * 70)
print(f"{'Variant':<26}  {'AUC':>12}  {'F1-macro':>14}  {'F1-trans':>12}")
print("-" * 68)
feat_n = {'RF-base (11 feats)': len(BASE_FEATS),
          'RF-temporal (8 feats)': len(ROLL_FEATS),
          'RF-full (19 feats)': len(ALL_FEATS)}
for name in ablation_configs:
    r = ablation_results[name]
    print(f"{name:<26}  {r['auc'][0]:.4f}±{r['auc'][1]:.4f}  "
          f"{r['f1_macro'][0]:.4f}±{r['f1_macro'][1]:.4f}  "
          f"{r['f1_trans'][0]:.4f}±{r['f1_trans'][1]:.4f}")
print("=" * 70)
print()

# Causal deltas — the key argument
f_base = ablation_results['RF-base (11 feats)']['f1_trans'][0]
f_temp = ablation_results['RF-temporal (8 feats)']['f1_trans'][0]
f_full = ablation_results['RF-full (19 feats)']['f1_trans'][0]

print("Causal attribution (F1-transitioning class):")
print(f"  Base features only:     {f_base:.4f}")
print(f"  Temporal features only: {f_temp:.4f}  (Δ vs base = {f_temp-f_base:+.4f})")
print(f"  Full feature set:       {f_full:.4f}  (Δ vs base = {f_full-f_base:+.4f})")
print()
if f_temp > f_base + 0.10:
    print("→ Temporal features are the dominant contributor to transitioning-class detection.")
    print("  Adding base features on top of temporal yields marginal improvement.")
else:
    print("→ Features are complementary. Both groups contribute to performance.")
print()
print("Paper narrative: temporal RoCoF/THD rolling statistics capture the PLL")
print("desynchronization dynamics that instantaneous measurements cannot resolve.")


In [ ]:
# ── Table V — Rolling Window Sensitivity (FIX 5 — reviewer objection defense) ─
# Addresses: "results too perfect / dataset too clean"
# Proves: performance degrades monotonically as window → 1 step (instantaneous)
# If F1-trans at window=1 ≈ SVM-inst → causal chain is complete.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

print("Rolling window sensitivity — F1-trans vs window size...")
print("(window=1 step = 5 min ≈ instantaneous; window=4 = 20 min = default)")
print()

WINDOWS = [1, 2, 4, 6, 8]
window_results = {}

for win in WINDOWS:
    # Rebuild rolling features with this window
    df_A_w = engineer_features(df_scA, 42,  window=win)
    df_B_w = engineer_features(df_scB, 137, window=win)
    df_C_w = engineer_features(df_scC, 999, window=win)
    df_D_w = engineer_features(df_scD, 314, window=win)

    def to_Xy_w(df_): return df_[ALL_FEATS].fillna(0).astype(float).values, le.transform(df_['control_mode'])
    X_B_w,y_B_w = to_Xy_w(df_B_w); X_C_w,y_C_w = to_Xy_w(df_C_w)
    X_A_w,y_A_w = to_Xy_w(df_A_w); X_D_w,y_D_w = to_Xy_w(df_D_w)
    Xtr_w = np.vstack([X_B_w, X_C_w]); ytr_w = np.concatenate([y_B_w, y_C_w])
    Xte_w = np.vstack([X_A_w, X_D_w]); yte_w = np.concatenate([y_A_w, y_D_w])

    f1_list = []
    for seed in SEEDS:
        sc_w = StandardScaler()
        Xtr_s = sc_w.fit_transform(Xtr_w); Xte_s = sc_w.transform(Xte_w)
        rf_w = RandomForestClassifier(n_estimators=200, class_weight='balanced',
            max_depth=18, min_samples_leaf=3, max_features='sqrt',
            random_state=seed, n_jobs=-1)
        rf_w.fit(Xtr_s, ytr_w)
        ypred_w = rf_w.predict(Xte_s)
        rep_w = classification_report(yte_w, ypred_w,
            labels=list(range(len(le.classes_))), target_names=le.classes_,
            output_dict=True, zero_division=0)
        f1_list.append(rep_w['transitioning']['f1-score'])
    window_results[win] = (np.mean(f1_list), np.std(f1_list))
    print(f"  Window={win} steps ({win*5:2d} min): F1-trans = {np.mean(f1_list):.4f}±{np.std(f1_list):.4f}")

print()
print("=" * 52)
print("TABLE V — TEMPORAL WINDOW SENSITIVITY (F1-transitioning)")
print("=" * 52)
print(f"{'Window (steps)':>16}  {'Min (min)':>10}  {'F1-trans':>12}")
print("-" * 44)
for win in WINDOWS:
    m, s = window_results[win]
    marker = " ← default" if win == ROLL_WIN else ""
    print(f"  {win:>14}  {win*5:>10}  {m:.4f}±{s:.4f}{marker}")
print("=" * 52)
print()
svm_inst_trans = svm_mean['inst']['f1_trans'][0]
win1_trans = window_results[1][0]
print(f"Window=1 F1-trans:   {win1_trans:.4f}")
print(f"SVM-inst F1-trans:   {svm_inst_trans:.4f}  (instantaneous baseline)")
if abs(win1_trans - svm_inst_trans) < 0.05:
    print("→ STRONG PROOF: RF with window=1 ≈ SVM-inst. The rolling window")
    print("  is the mechanism — not the Random Forest architecture.")
print()
print("Physical interpretation: at window=1 (one 5-min snapshot), the RoCoF")
print("rolling statistics collapse to instantaneous measurements, eliminating")
print("the PLL desynchronization signature that distinguishes transitioning.")


In [ ]:
# ── Noise Stress Test — reviewer defense: "dataset too clean" ─────────────────
# Applies non-Gaussian noise + missing data to the test set.
# RF trained on clean data; evaluated on degraded test.
# Shows robustness and answers the "real-world" objection.
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

# Train the best RF (seed=42)
sc_ns = StandardScaler()
Xtr_ns = sc_ns.fit_transform(X_train)
rf_ns  = RandomForestClassifier(n_estimators=400, class_weight='balanced',
    max_depth=18, min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)
rf_ns.fit(Xtr_ns, y_train)

noise_results = {}
print("Noise stress test — RF trained on clean data, evaluated on degraded test:")
print()

stress_configs = {
    'Clean (baseline)':          (0.00, 0.00),
    'Low noise (σ=10%)':         (0.10, 0.00),
    'Medium noise (σ=20%)':      (0.20, 0.00),
    'Missing 5% (uniform)':      (0.05, 0.05),
    'Missing 10% (uniform)':     (0.10, 0.10),
    'High noise + missing 10%':  (0.20, 0.10),
}

rng_stress = np.random.default_rng(42)
for label, (noise_sigma, missing_frac) in stress_configs.items():
    X_te_stress = X_test.copy()
    # Non-Gaussian noise: uniform distribution (harder than Gaussian)
    if noise_sigma > 0:
        noise = rng_stress.uniform(
            -noise_sigma * X_test.std(axis=0),
             noise_sigma * X_test.std(axis=0),
             X_test.shape)
        X_te_stress = X_te_stress + noise
    # Missing data: zero-out random entries
    if missing_frac > 0:
        mask = rng_stress.random(X_te_stress.shape) < missing_frac
        X_te_stress[mask] = 0.0

    Xte_s = sc_ns.transform(X_te_stress)
    ypred = rf_ns.predict(Xte_s)
    rep = classification_report(y_test, ypred,
        labels=list(range(len(le.classes_))), target_names=le.classes_,
        output_dict=True, zero_division=0)
    f1_m  = rep['macro avg']['f1-score']
    f1_tr = rep['transitioning']['f1-score']
    noise_results[label] = (f1_m, f1_tr)
    print(f"  {label:<30}: F1-macro={f1_m:.4f}  F1-trans={f1_tr:.4f}")

print()
baseline_f1   = noise_results['Clean (baseline)'][0]
hardest_f1    = noise_results['High noise + missing 10%'][0]
degradation   = baseline_f1 - hardest_f1
print(f"F1-macro degradation (clean → hardest): {degradation:.4f} ({degradation/baseline_f1:.1%} relative)")
print()
print("Paper text: The classifier retains [X]% of clean-data performance under")
print("non-Gaussian sensor noise (σ=20%) and 10% missing data, confirming")
print("robustness to measurement uncertainty typical of industrial SCADA systems.")


In [ ]:
# ── Table IV — Cross-Scenario Generalization ─────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report

cross_configs = [
    ('A → A (within)',      X_A, y_A, X_A, y_A),
    ('B → B (within)',      X_B, y_B, X_B, y_B),
    ('C → C (within)',      X_C, y_C, X_C, y_C),
    ('D → D (within)',      X_D, y_D, X_D, y_D),
    ('B → A (cross)',       X_B, y_B, X_A, y_A),
    ('C → D (cross)',       X_C, y_C, X_D, y_D),
    ('B+C → A+D (paper)',   X_train, y_train, X_test, y_test),
]

cross_results = []
print("Cross-scenario generalization (RF, seed=42)...")
for label, Xtr, ytr, Xte, yte in cross_configs:
    sc_cs = StandardScaler()
    Xtr_s = sc_cs.fit_transform(Xtr); Xte_s = sc_cs.transform(Xte)
    rf_cs = RandomForestClassifier(n_estimators=200, class_weight='balanced',
        max_depth=18, min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)
    rf_cs.fit(Xtr_s, ytr)
    ypred = rf_cs.predict(Xte_s); yprob = rf_cs.predict_proba(Xte_s)
    try:
        auc = roc_auc_score(yte, yprob, multi_class='ovr', average='macro',
                            labels=list(range(len(le.classes_))))
    except ValueError:
        auc = float('nan')
    rep = classification_report(yte, ypred,
        labels=list(range(len(le.classes_))), target_names=le.classes_,
        output_dict=True, zero_division=0)
    cross_results.append({'label': label, 'auc': auc,
        'f1_macro': rep['macro avg']['f1-score'],
        'f1_trans': rep['transitioning']['f1-score']})

print()
print("=" * 68)
print("TABLE IV — CROSS-SCENARIO GENERALIZATION (RF, seed=42)")
print("=" * 68)
print(f"{'Experiment':<24}  {'Type':>9}  {'AUC':>8}  {'F1-macro':>10}  {'F1-trans':>10}")
print("-" * 68)
for r in cross_results:
    t = 'within' if 'within' in r['label'] else 'cross'
    auc_s = f"{r['auc']:.4f}" if not (isinstance(r['auc'], float) and r['auc'] != r['auc']) else '  —   '
    print(f"{r['label']:<24}  {t:>9}  {auc_s:>8}  {r['f1_macro']:>10.4f}  {r['f1_trans']:>10.4f}")
print("=" * 68)

within_f1 = np.mean([r['f1_macro'] for r in cross_results if 'within' in r['label']])
cross_f1  = np.mean([r['f1_macro'] for r in cross_results if 'within' not in r['label']])
retention = cross_f1 / within_f1 if within_f1 > 0 else 0
print(f"\nRetention: {retention:.1%} ({cross_f1:.4f} cross vs {within_f1:.4f} within)")
print()
print("Physical interpretation:")
print("  B→A (cross-scenario): model trained on mixed/transitioning tested on")
print("    GFL-dominant scenario. High retention indicates the rolling RoCoF")
print("    features capture the GFL steady-state dynamics not seen in training.")
print("  C→D (cross-scenario): model trained on transitioning-heavy tested on")
print("    stressed grid. Tests generalization to faster islanding dynamics.")
print(f"  B+C→A+D (paper): {retention:.1%} retention — generalization across both extremes.")


In [ ]:
# ── SHAP: Why RF outperforms ROCOF-threshold on transitioning class ──────────
try:
    import shap

    # Final RF model on seed=42
    sc_final = StandardScaler()
    X_tr_f, X_te_f, y_tr_f, y_te_f = X_train, X_test, y_train, y_test
    X_tr_fs = sc_final.fit_transform(X_tr_f)
    X_te_fs  = sc_final.transform(X_te_f)

    rf_final = RandomForestClassifier(
        n_estimators=400, class_weight='balanced', max_depth=18,
        min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)
    rf_final.fit(X_tr_fs, y_tr_f)

    rng  = np.random.default_rng(42)
    idx  = rng.choice(len(X_te_fs), size=min(600, len(X_te_fs)), replace=False)
    expl = shap.TreeExplainer(rf_final)
    sv   = expl.shap_values(X_te_fs[idx])

    # ── Handle both SHAP return formats ──────────────────────────────────────
    # Old SHAP (<0.42): sv is a list of arrays, one per class
    #   sv[class_idx]  → shape (n_samples, n_features)  ✅
    # New SHAP (>=0.42): sv is a single 3D array
    #   sv             → shape (n_samples, n_features, n_classes)
    #   sv[:, :, class_idx] → shape (n_samples, n_features)  ✅
    import numpy as np

    trans_idx = list(le.classes_).index('transitioning')

    if isinstance(sv, list):
        # Old format — list of (n_samples, n_features) arrays
        sv_trans = sv[trans_idx]
    elif sv.ndim == 3:
        # New format — (n_samples, n_features, n_classes)
        sv_trans = sv[:, :, trans_idx]
    else:
        # Fallback: binary or unexpected shape
        sv_trans = sv

    print(f"SHAP version format: {'list' if isinstance(sv, list) else f'{sv.ndim}D array'}")
    print(f"sv_trans shape: {sv_trans.shape}  (expected: n_samples x {len(ALL_FEATS)})")

    # ── Feature labels ────────────────────────────────────────────────────────
    LABELS = {
        'rocof_hz_per_s':          'ROCOF (Hz/s)',
        'thd_percent':             'THD (%)',
        'freq_deviation_hz':       'Freq Deviation (Hz)',
        'voltage_deviation_pu':    'Voltage Dev (p.u.)',
        'output_active_power_kw':  'Active Power (kW)',
        'output_reactive_power_kvar': 'Reactive Power (kVAr)',
        'output_voltage_v':        'Voltage (V)',
        'output_frequency_hz':     'Output Frequency (Hz)',
        'dc_link_voltage_v':       'DC Link Voltage (V)',
        'junction_temp_c':         'Junction Temp (C)',
        'efficiency':              'Efficiency',
        'gfl_pll_locked':          'PLL Locked',
        'rocof_roll_mean':         'ROCOF Roll Mean <-',
        'rocof_roll_std':          'ROCOF Roll Std <-',
        'rocof_roll_max':          'ROCOF Roll Max <-',
        'freq_roll_std':           'Freq Roll Std <-',
        'freq_roll_mean':          'Freq Roll Mean <-',
        'thd_roll_mean':           'THD Roll Mean <-',
        'power_roll_std':          'Power Roll Std <-',
        'volt_roll_std':           'Volt Roll Std <-',
    }
    feat_labels = [LABELS.get(f, f) for f in ALL_FEATS]

    shap_trans = (pd.Series(np.abs(sv_trans).mean(axis=0), index=feat_labels)
                    .sort_values(ascending=False))

    # ── Plots ─────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Top-10 features for TRANSITIONING class
    top10 = shap_trans.head(10).sort_values()
    roll_mask  = ['<-' in n for n in top10.index]
    bar_colors = ['#10B981' if m else '#3B82F6' for m in roll_mask]
    axes[0].barh(top10.index, top10.values, color=bar_colors, alpha=0.9)
    axes[0].set_title(
        'SHAP - Top Features for TRANSITIONING class\n'
        '(green = rolling temporal features)',
        fontweight='bold')
    axes[0].set_xlabel('Mean |SHAP value|')

    # Right: Rolling vs instantaneous importance share
    roll_imp = shap_trans[[n for n in shap_trans.index if '<-' in n]].sum()
    inst_imp = shap_trans[[n for n in shap_trans.index if '<-' not in n]].sum()
    total    = roll_imp + inst_imp

    pct_roll = roll_imp / total
    pct_inst = inst_imp / total
    label_roll = f'Rolling features\n(proposed) {pct_roll:.0%}'
    label_inst = f'Instantaneous\n(SVM baseline) {pct_inst:.0%}'

    axes[1].pie(
        [roll_imp, inst_imp],
        labels=[label_roll, label_inst],
        colors=['#10B981', '#3B82F6'],
        autopct='%1.0f%%',
        startangle=90,
        textprops=dict(color='white')
    )
    axes[1].set_title(
        'SHAP importance share:\nRolling vs Instantaneous features',
        fontweight='bold')

    plt.suptitle(
        'Why temporal rolling features matter: SHAP on TRANSITIONING class',
        fontweight='bold', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('shap_transitioning.png', dpi=150, bbox_inches='tight',
                facecolor='#0F172A')
    plt.show()

    print("Key insight for paper:")
    print(f"  Rolling features account for {pct_roll:.0%} of SHAP importance")
    print(f"  on the transitioning class -- temporal dynamics are the key discriminator.")
    print(f"  This is precisely what the SVM (instantaneous features only) misses.")

except ImportError:
    print("SHAP not installed -- skipping. Run: pip install shap")


In [ ]:
# ── 5-Fold Stratified Cross-Validation (RF) ──────────────────────────────────
# roc_auc_ovr is dropped from cross_validate because StratifiedKFold can
# produce folds missing a rare class (transitioning), causing NaN.
# Instead: compute AUC manually per fold using cross_val_predict.
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sc_cv = StandardScaler()
X_all_s = sc_cv.fit_transform(X_train)
y_cv    = y_train

rf_cv_model = RandomForestClassifier(
    n_estimators=400, class_weight='balanced', max_depth=18,
    min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)

# F1-macro per fold (handles missing classes via zero_division)
f1_scores = cross_val_score(
    rf_cv_model, X_all_s, y_cv, cv=cv,
    scoring='f1_macro', n_jobs=-1)

# AUC per fold — manual loop to skip folds missing a class
auc_scores = []
for train_idx, test_idx in cv.split(X_all_s, y_cv):
    rf_fold = RandomForestClassifier(
        n_estimators=400, class_weight='balanced', max_depth=18,
        min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)
    rf_fold.fit(X_all_s[train_idx], y_cv[train_idx])
    y_prob_fold = rf_fold.predict_proba(X_all_s[test_idx])
    classes_in_fold = np.unique(y_cv[test_idx])
    if len(classes_in_fold) < 2:
        continue  # skip fold with single class — cannot compute AUC
    try:
        auc = roc_auc_score(
            y_cv[test_idx], y_prob_fold,
            multi_class='ovr', average='macro',
            labels=list(range(len(le.classes_))))
        auc_scores.append(auc)
    except ValueError:
        pass  # fold has missing class — skip

auc_arr = np.array(auc_scores) if auc_scores else np.array([float('nan')])

print("5-Fold Stratified Cross-Validation (RF, all Dataset A):")
print(f"  AUC OvR macro : {auc_arr.mean():.4f} ± {auc_arr.std():.4f}"
      f"  (folds with all classes: {len(auc_scores)}/5)")
print(f"  F1 macro      : {f1_scores.mean():.4f} ± {f1_scores.std():.4f}")
print(f"  Fold AUCs     : {np.round(auc_arr, 4).tolist()}")
print(f"  Fold F1s      : {np.round(f1_scores, 4).tolist()}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION IV-D — Latency Optimization for Industrial Relay Deployment
# Target: < 20 ms (numerical protective relay requirement)
# Current baseline: ~29 ms (400 trees, depth=18) → ❌ relay target
#
# Strategy: sweep (n_estimators, max_depth) and identify the Pareto front
# between F1-macro and latency. Find the smallest model that stays within
# the relay target while maximizing accuracy.
# ══════════════════════════════════════════════════════════════════════════════
import time
import tempfile, os, joblib as jlb
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

IEEE_LIMIT_MS  = 2000.0   # ms — IEEE 1547-2018 §6.5.1
RELAY_TARGET   = 20.0     # ms — numerical protective relay
N_TRIALS       = 5000     # timing trials per config
BATCH_SIZES    = [1, 4, 6, 8, 16]

sc_lat = StandardScaler()
X_tr_ls = sc_lat.fit_transform(X_train)
X_te_ls  = sc_lat.transform(X_test)

def measure_latency(model, X_sample, n_trials=N_TRIALS):
    # Warm-up
    for _ in range(100):
        model.predict(X_sample[0:1])
    t0 = time.perf_counter()
    for _ in range(n_trials):
        model.predict(X_sample[0:1])
    return (time.perf_counter() - t0) / n_trials * 1000  # ms

def model_size_kb(model):
    tmp = tempfile.NamedTemporaryFile(suffix=".pkl", delete=False)
    tmp.close()
    try:
        jlb.dump(model, tmp.name)
        return os.path.getsize(tmp.name) / 1024
    finally:
        os.unlink(tmp.name)

# ── Full sweep: n_estimators × max_depth ─────────────────────────────────────
print("Sweeping (n_estimators × max_depth) — this takes ~3-4 minutes...")
print()

CONFIGS = [
    # (n_estimators, max_depth, label)
    (400, 18, "baseline (full)"),
    (200, 18, "200 trees"),
    (100, 18, "100 trees"),
    ( 50, 18,  "50 trees"),
    ( 25, 18,  "25 trees"),
    (100, 12, "100t d12"),
    (100,  8, "100t d8"),
    ( 75, 10,  "75t d10"),
    ( 50, 10,  "50t d10"),
    ( 50,  8,   "50t d8"),
    ( 30,  8,   "30t d8"),
    ( 20,  8,   "20t d8"),
    ( 15,  6,   "15t d6"),
]

results_opt = []
print(f"{'Config':<22}  {'Trees':>6}  {'Depth':>6}  {'Lat(ms)':>9}  {'<20ms':>6}  {'F1-mac':>8}  {'F1-GFL':>8}  {'F1-tr':>8}  {'Size(KB)':>9}")
print("-" * 100)

for n_est, max_d, label in CONFIGS:
    rf = RandomForestClassifier(
        n_estimators=n_est, max_depth=max_d,
        class_weight="balanced", min_samples_leaf=3,
        max_features="sqrt", random_state=42, n_jobs=1)
    rf.fit(X_tr_ls, y_train)

    y_pred = rf.predict(X_te_ls)
    rep = classification_report(y_test, y_pred,
        labels=list(range(len(le.classes_))), target_names=le.classes_,
        output_dict=True, zero_division=0)
    f1m  = rep["macro avg"]["f1-score"]
    f1g  = rep["GFL"]["f1-score"]
    f1tr = rep["transitioning"]["f1-score"]

    lat_ms   = measure_latency(rf, X_te_ls)
    sz_kb    = model_size_kb(rf)
    ok_relay = lat_ms < RELAY_TARGET

    results_opt.append({
        "label": label, "n_est": n_est, "depth": max_d,
        "lat_ms": lat_ms, "f1_macro": f1m, "f1_gfl": f1g, "f1_trans": f1tr,
        "size_kb": sz_kb, "relay_ok": ok_relay,
    })

    relay_str = "✅" if ok_relay else "❌"
    print(f"  {label:<20}  {n_est:>6}  {max_d:>6}  {lat_ms:>9.2f}  {relay_str:>6}  "
          f"{f1m:>8.4f}  {f1g:>8.4f}  {f1tr:>8.4f}  {sz_kb:>9.1f}")

# ── Pareto front: relay_ok AND best F1 ───────────────────────────────────────
print()
relay_ok = [r for r in results_opt if r["relay_ok"]]
if relay_ok:
    best = max(relay_ok, key=lambda r: r["f1_macro"])
    print(f"Best relay-compliant config: {best['label']}")
    print(f"  n_estimators={best['n_est']}, max_depth={best['depth']}")
    print(f"  Latency={best['lat_ms']:.2f} ms  F1-macro={best['f1_macro']:.4f}  "
          f"F1-trans={best['f1_trans']:.4f}  Size={best['size_kb']:.0f} KB")
    baseline = results_opt[0]
    print(f"  F1 penalty vs baseline: {best['f1_macro'] - baseline['f1_macro']:+.4f}")
    print(f"  Latency reduction: {baseline['lat_ms']:.1f} ms → {best['lat_ms']:.1f} ms "
          f"({100*(1-best['lat_ms']/baseline['lat_ms']):.0f}% faster)")
else:
    best = min(results_opt, key=lambda r: r["lat_ms"])
    print(f"No config meets 20ms. Closest: {best['label']} @ {best['lat_ms']:.2f}ms")


In [ ]:
# ── Paper Table III: Latency Comparison (baseline vs optimized) ──────────────
import time
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

IEEE_LIMIT_MS = 2000.0
RELAY_TARGET  = 20.0

# Best relay-compliant config from sweep
best_cfg = max([r for r in results_opt if r["relay_ok"]],
               key=lambda r: r["f1_macro"]) if any(r["relay_ok"] for r in results_opt)            else min(results_opt, key=lambda r: r["lat_ms"])

# Rebuild both models for batch latency
sc_a = StandardScaler()
X_tr_a = sc_a.fit_transform(X_train); X_te_a = sc_a.transform(X_test)

rf_base = RandomForestClassifier(n_estimators=400, max_depth=18,
    class_weight="balanced", min_samples_leaf=3, max_features="sqrt",
    random_state=42, n_jobs=1)
rf_base.fit(X_tr_a, y_train)

rf_opt = RandomForestClassifier(n_estimators=best_cfg["n_est"],
    max_depth=best_cfg["depth"], class_weight="balanced",
    min_samples_leaf=3, max_features="sqrt", random_state=42, n_jobs=1)
rf_opt.fit(X_tr_a, y_train)

def batch_latency(model, X, bs, n_reps=2000):
    batch = X[:bs]
    for _ in range(50): model.predict(batch)
    t0 = time.perf_counter()
    for _ in range(n_reps): model.predict(batch)
    return (time.perf_counter() - t0) / n_reps * 1000

print("=" * 80)
print("TABLE III — INFERENCE LATENCY: BASELINE vs RELAY-OPTIMIZED")
print(f"Baseline: 400 trees, depth=18   |   Optimized: {best_cfg['n_est']} trees, depth={best_cfg['depth']}")
print("=" * 80)
print(f"{'Batch':>6}  {'Baseline (ms)':>14}  {'<IEEE':>6}  {'<relay':>7}  "
      f"{'Optimized (ms)':>15}  {'<IEEE':>6}  {'<relay':>7}")
print("-" * 70)

for bs in [1, 4, 6, 8, 16]:
    lb = batch_latency(rf_base, X_te_a, bs)
    lo = batch_latency(rf_opt,  X_te_a, bs)
    ib = "✅" if lb < IEEE_LIMIT_MS else "❌"
    ir = "✅" if lb < RELAY_TARGET  else "❌"
    ob = "✅" if lo < IEEE_LIMIT_MS else "❌"
    or_ = "✅" if lo < RELAY_TARGET else "❌"
    print(f"  {bs:>6}  {lb:>14.2f}  {ib:>6}  {ir:>7}  {lo:>15.2f}  {ob:>6}  {or_:>7}")

print("=" * 80)
print()

# F1 comparison
from sklearn.metrics import classification_report
y_base = rf_base.predict(X_te_a)
y_opt  = rf_opt.predict(X_te_a)
r_b = classification_report(y_test, y_base,
    labels=list(range(len(le.classes_))), target_names=le.classes_,
    output_dict=True, zero_division=0)
r_o = classification_report(y_test, y_opt,
    labels=list(range(len(le.classes_))), target_names=le.classes_,
    output_dict=True, zero_division=0)

print(f"F1-macro:   baseline={r_b['macro avg']['f1-score']:.4f}  "
      f"optimized={r_o['macro avg']['f1-score']:.4f}  "
      f"delta={r_o['macro avg']['f1-score']-r_b['macro avg']['f1-score']:+.4f}")
print(f"F1-trans:   baseline={r_b['transitioning']['f1-score']:.4f}  "
      f"optimized={r_o['transitioning']['f1-score']:.4f}")
print(f"F1-GFL:     baseline={r_b['GFL']['f1-score']:.4f}  "
      f"optimized={r_o['GFL']['f1-score']:.4f}")
print()
sz_b = model_size_kb(rf_base)
sz_o = model_size_kb(rf_opt)
print(f"Model size: baseline={sz_b:.0f} KB  optimized={sz_o:.0f} KB  "
      f"reduction={100*(1-sz_o/sz_b):.0f}%")
print()
print("Paper paragraph (Section IV-D):")
lat_opt_single = best_cfg["lat_ms"]
speedup_ieee   = IEEE_LIMIT_MS / lat_opt_single
speedup_relay  = RELAY_TARGET  / lat_opt_single
print(f"  The optimized classifier ({best_cfg['n_est']} estimators, depth={best_cfg['depth']})")
print(f"  achieves {lat_opt_single:.1f} ms single-sample latency,")
print(f"  satisfying both the IEEE 1547-2018 §6.5.1 limit ({speedup_ieee:.0f}× margin)")
print(f"  and the 20 ms numerical relay target ({speedup_relay:.1f}× margin),")
print(f"  at a cost of {r_o['macro avg']['f1-score']-r_b['macro avg']['f1-score']:+.4f} F1-macro relative to the full model.")
print(f"  Model size is reduced from {sz_b:.0f} KB to {sz_o:.0f} KB ({100*(1-sz_o/sz_b):.0f}% reduction).")


In [ ]:
# ── Fig: Pareto Front — F1-macro vs Inference Latency ────────────────────────
# Generates figures/pareto_latency_f1.pdf for the paper
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os

os.makedirs("../paper/figures", exist_ok=True)

# Data from latency sweep (results_opt populated in previous cell)
configs_label = [r["label"]    for r in results_opt]
latencies     = [r["lat_ms"]   for r in results_opt]
f1_macros     = [r["f1_macro"] for r in results_opt]
f1_trans_vals = [r["f1_trans"] for r in results_opt]
relay_ok      = [r["relay_ok"] for r in results_opt]

fig, ax = plt.subplots(figsize=(5.5, 4.0))

# Color by relay compliance
colors = ["#2196F3" if ok else "#F44336" for ok in relay_ok]
sizes  = [80 if ok else 50 for ok in relay_ok]

sc = ax.scatter(latencies, f1_macros, c=colors, s=sizes, zorder=5, alpha=0.85)

# Annotate key points
best_idx   = next(i for i,r in enumerate(results_opt) if r["label"] == "200 trees")
worst_idx  = 0  # baseline

ax.scatter([latencies[best_idx]], [f1_macros[best_idx]],
           marker="*", s=280, c="#2196F3", zorder=6, edgecolors="k", linewidth=0.8,
           label="Pareto-optimal (200 trees, 14.2 ms)")
ax.scatter([latencies[worst_idx]], [f1_macros[worst_idx]],
           marker="s", s=80, c="#F44336", zorder=6, edgecolors="k", linewidth=0.8,
           label="Baseline (400 trees, 28.9 ms)")

# Annotate n_estimators for key points
for r in results_opt:
    if r["n_est"] in [25, 50, 100, 200, 400] and r["depth"] == 18:
        ax.annotate(
            f"{r['n_est']}t",
            (r["lat_ms"], r["f1_macro"]),
            textcoords="offset points", xytext=(4, 3),
            fontsize=7, color="#333333"
        )

# Relay target line
ax.axvline(x=20, color="#FF5722", linestyle="--", linewidth=1.2, alpha=0.8,
           label="20 ms relay target")

# F1 floor line
ax.axhline(y=0.980, color="#9E9E9E", linestyle=":", linewidth=0.8, alpha=0.7,
           label="F1-macro = 0.980")

# Shade relay-compliant region
ax.axvspan(0, 20, alpha=0.06, color="#2196F3")

ax.set_xlabel("Single-sample latency (ms)", fontsize=10)
ax.set_ylabel("F1-macro", fontsize=10)
ax.set_title("Pareto Front: F1-macro vs. Inference Latency", fontsize=10, fontweight="bold")
ax.legend(fontsize=7, loc="lower right")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0.94)
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.tick_params(labelsize=8)

# F1-trans annotation
ax.text(0.03, 0.04,
        "F1-trans = 1.000 for all configs\nwith >=25 estimators",
        transform=ax.transAxes, fontsize=7, color="#555555",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

plt.tight_layout()
outpath = "../paper/figures/pareto_latency_f1.pdf"
plt.savefig(outpath, dpi=200, bbox_inches="tight")
plt.savefig(outpath.replace(".pdf", ".png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved: {outpath}")
print(f"Also:  {outpath.replace('.pdf','.png')}")


In [ ]:
# ── Save models + results ─────────────────────────────────────────────────────
import os, joblib

os.makedirs('../ml', exist_ok=True)
sc_save = StandardScaler()
rf_save = RandomForestClassifier(n_estimators=400, class_weight='balanced',
    max_depth=18, min_samples_leaf=3, max_features='sqrt', random_state=42, n_jobs=-1)
rf_save.fit(sc_save.fit_transform(X_train), y_train)

joblib.dump(rf_save, '../ml/gfm_classifier.pkl')
joblib.dump(sc_save, '../ml/gfm_scaler.pkl')
joblib.dump(le,      '../ml/gfm_label_encoder.pkl')
with open('../ml/gfm_features.json', 'w') as f_:
    json.dump(ALL_FEATS, f_)

multi_seed_export = {
    'RF':  {k: [float(v) for v in vals] for k, vals in rf_res.items() if k != 'preds'},
    'SVM-inst': {k: [float(v) for v in vals] for k, vals in svm_results['inst'].items()},
    'SVM-roll': {k: [float(v) for v in vals] for k, vals in svm_results['roll'].items()},
}
if XGB_OK:
    multi_seed_export['XGB'] = {k: [float(v) for v in vals] for k, vals in xgb_res.items()}
with open('../ml/gfm_multiseed_results.json', 'w') as f_:
    json.dump(multi_seed_export, f_, indent=2)

print("Saved: ml/gfm_classifier.pkl, gfm_scaler.pkl, gfm_label_encoder.pkl")
print("       ml/gfm_features.json, ml/gfm_multiseed_results.json")
print()
print("Dataset parameters (Section III):")
print("  Scenario A: p=0.08, seed=42  — GFL-dominant  [TEST]")
print("  Scenario B: p=0.15, seed=137 — mixed          [TRAIN]")
print("  Scenario C: p=0.35, seed=999 — trans-heavy    [TRAIN]")
print("  Scenario D: p=0.55, seed=314 — GFM-dominant  [TEST]")
print("  Split: train B+C → test A+D (scenario-based, zero leakage)")
print("  Rolling window: 4 steps = 20 min")
print("  gfl_pll_locked: removed (label proxy)")
print("  Gaussian noise: sigma=5% (sensor uncertainty)")
